## Global Utilities Notebook (`common/Globals`)

### Purpose
The `Globals` notebook contains shared helper functions and constants used across Bronze ingestion notebooks.  
It ensures ingestion logic remains reusable, consistent, and metadata-driven.

---

### Key Features

#### 1) Standard Bronze Audit Columns (Reusable DDL)
A global SQL snippet is defined to ensure every Bronze table contains consistent audit metadata:

- `loaded_at` (timestamp when the row was ingested)
- `updated_at` (timestamp for last update in Bronze, initially same as loaded_at)
- `load_dt` (date partitioning column for filtering and reconciliation)
- `source_file` (file-level lineage using `_metadata.file_name`)

---

#### 2) YAML Config Loader
A reusable function loads the ingestion configuration from a YAML file and returns it as a Python dictionary.  
This enables metadata-driven ingestion instead of hardcoding table logic.

---

#### 3) Fully Qualified Table Name Builder
A helper function constructs fully qualified table names in Unity Catalog format:

`<catalog>.<schema>.<table>`

This ensures consistent and unambiguous table references across environments.

---

#### 4) Source Path Builder (UC Volume)
A helper function constructs the raw file source path by combining:

- the base UC volume path (`raw_volume`)
- the dataset folder name (`source_subfolder`)

This allows ingestion to work across multiple datasets using a single notebook.

---

#### 5) Bronze Table DDL Generator
A reusable function generates a `CREATE TABLE IF NOT EXISTS` SQL statement for Bronze tables.

Key design decisions:
- All raw columns are created as `STRING` to prevent ingestion failures due to dirty input data
- Audit columns are appended consistently across all tables
- An optional folder-level lineage column (`source`) can be added when configured

---

### Outcome
By centralizing common logic in `Globals`, the Bronze layer achieves:

- consistent audit and lineage standards  
- reusable ingestion patterns  
- scalable onboarding of new datasets by only updating YAML configuration  


In [0]:
# ============================================================
# common_globals
# Shared utilities + global variables for Bronze ingestion
# ============================================================

import yaml

# -------------------------
# Global audit columns
# -------------------------
AUDIT_COLS_BASE_DDL = """
  loaded_at TIMESTAMP,
  updated_at TIMESTAMP,
  load_dt DATE,
  source_file STRING
"""

def load_config(config_path: str) -> dict:
    """
    Loads YAML config from repo path.
    """


def load_config(config_path: str) -> dict:
    """
    Loads YAML config from a UC Volume path.
    Example:
    /Volumes/workspace/default/coffee_raw_volume/resources/configs/bronze_config.yml
    """
    with open(config_path, "r") as f:
        return yaml.safe_load(f)

    # For Volumes or normal paths
    with open(config_path, "r") as f:
        return yaml.safe_load(f)


def build_table_fqn(catalog: str, schema: str, table: str) -> str:
    """
    Builds fully qualified table name.
    Example: coffee.bronze.users
    """
    return f"{catalog}.{schema}.{table}"

def build_source_path(raw_volume: str, subfolder: str) -> str:
    """
    Builds volume path for raw files.
    Example:
    raw_volume=/Volumes/workspace/default/coffee_raw_volume
    subfolder=users
    -> /Volumes/workspace/default/coffee_raw_volume/users/
    """
    return f"{raw_volume}/{subfolder}/"

def build_create_table_sql(
    table_fqn: str,
    raw_columns: list[str],
    add_source_column: bool = False
) -> str:
    """
    Creates CREATE TABLE SQL for Bronze table.
    - All raw columns are STRING
    - Adds standard audit columns
    - Optionally adds 'source' column (folder-level lineage)
    """
    cols_ddl = ",\n  ".join([f"{c} STRING" for c in raw_columns])

    extra_source = ""
    if add_source_column:
        extra_source = ",\n  source STRING"

    return f"""
    CREATE TABLE IF NOT EXISTS {table_fqn} (
      {cols_ddl},
      {AUDIT_COLS_BASE_DDL}{extra_source}
    )
    USING DELTA
    """
